# Breast Cancer Detection — ML Classification Project

In this notebook I'm building a model that classifies breast tumors as **Malignant** or **Benign**.
The dataset comes from sklearn — 569 patients, 30 numeric features from biopsy images.

**Plan:**
1. Load and explore the data
2. Feature engineering
3. Preprocessing pipeline
4. Train and compare models
5. Save the best one

In [ ]:
# Step 1: Imports
# standard data science stuff
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# load the built-in dataset from sklearn
from sklearn.datasets import load_breast_cancer

# splitting data and cross-validation
from sklearn.model_selection import train_test_split, cross_val_score

# preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# all the models I'll compare
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# metrics
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
# Step 2: Load Dataset
# sklearn has this built in — no downloading needed
# target: 0 = Malignant, 1 = Benign

data = load_breast_cancer()

# put it into a dataframe so it's easier to work with
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print(f"Dataset shape: {df.shape}")
print(f"Classes: {data.target_names}")
df.sample(5)


In [ ]:
# Step 3: EDA — check class distribution
# want to see if the dataset is balanced

plt.figure(figsize=(6, 4))
df['target'].value_counts().plot(kind='bar', color=['#e74c3c', '#2ecc71'], edgecolor='black')
plt.title('Class Distribution (0 = Malignant, 1 = Benign)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Step 4: Scatter plot — does mean radius separate the classes?
# coloring by class to see if there's a visible cluster pattern

plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    df['mean radius'],
    df['mean texture'],
    c=df['target'],
    cmap='RdYlGn',
    alpha=0.7,
    edgecolors='k',
    linewidths=0.3
)
plt.colorbar(scatter, label='0=Malignant / 1=Benign')
plt.xlabel('Mean Radius')
plt.ylabel('Mean Texture')
plt.title('Mean Radius vs Mean Texture')
plt.tight_layout()
plt.show()


In [ ]:
# Step 5: Feature Engineering
# creating two extra columns from existing features
# mainly to practice using OrdinalEncoder and OneHotEncoder later

# ordinal: tumor size category based on radius bins
df['tumor_size_category'] = pd.cut(
    df['mean radius'],
    bins=[0, 12, 18, 30],
    labels=['Small', 'Medium', 'Large']
)

# nominal: texture type based on whether it's above or below the median
df['texture_type'] = np.where(
    df['mean texture'] > df['mean texture'].median(),
    'Rough',
    'Smooth'
)

print("New features:")
df[['tumor_size_category', 'texture_type']].head()


In [ ]:
# Step 6: Train/Test Split
# 80/20 split, stratify=y keeps class ratios the same in both sets

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")


In [ ]:
# Step 7: Preprocessing Pipeline
# using ColumnTransformer to apply different transformations to different column types
# - numerical  -> StandardScaler (zero mean, unit variance)
# - ordinal    -> OrdinalEncoder (respects Small < Medium < Large order)
# - nominal    -> OneHotEncoder (no ordering, just 0s and 1s)

numerical_features = X.select_dtypes(include=np.number).columns.tolist()
ordinal_features   = ['tumor_size_category']
nominal_features   = ['texture_type']

# need to specify the order for OrdinalEncoder
tumor_order = [['Small', 'Medium', 'Large']]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(),                        numerical_features),
        ('ord', OrdinalEncoder(categories=tumor_order),  ordinal_features),
        ('nom', OneHotEncoder(handle_unknown='ignore'),  nominal_features)
    ]
)

print("Preprocessor ready.")


In [ ]:
# Step 8: Train and Compare Models
# wrapping each model in a Pipeline so preprocessing is always applied consistently
# using 5-fold CV on the training set for a more reliable accuracy estimate

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Ridge (L2)':          LogisticRegression(penalty='l2', max_iter=1000),
    'Lasso (L1)':          LogisticRegression(penalty='l1', solver='saga', max_iter=1000),
    'SVM':                 SVC(probability=True),
    'KNN':                 KNeighborsClassifier(),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42)
}

best_model      = None
best_model_name = None
best_accuracy   = 0.0
results         = []

for name, model in models.items():
    # full pipeline: preprocess -> model
    pipe = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    # 5-fold cross-validation on training data
    cv_mean = cross_val_score(pipe, X_train, y_train, cv=5).mean()

    # train on full training set, evaluate on held-out test set
    pipe.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, pipe.predict(X_test))

    results.append({'Model': name, 'CV Accuracy': cv_mean, 'Test Accuracy': test_acc})
    print(f"{name:25s}  CV: {cv_mean:.4f}  |  Test: {test_acc:.4f}")

    # track the best one
    if test_acc > best_accuracy:
        best_accuracy   = test_acc
        best_model      = pipe
        best_model_name = name

print(f"\nBest: {best_model_name} ({best_accuracy:.4f})")


In [ ]:
# Step 9: Results Table
# clean summary sorted by test accuracy

results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
results_df['CV Accuracy']   = results_df['CV Accuracy'].apply(lambda x: f"{x*100:.2f}%")
results_df['Test Accuracy'] = results_df['Test Accuracy'].apply(lambda x: f"{x*100:.2f}%")
results_df.reset_index(drop=True, inplace=True)
print(results_df.to_string(index=False))


In [ ]:
# Step 10: Classification Report
# precision, recall, F1 — important here because missing a cancer (false negative) is costly

y_pred_best = best_model.predict(X_test)
print(f"Classification Report — {best_model_name}")
print('=' * 50)
print(classification_report(y_test, y_pred_best, target_names=['Malignant', 'Benign']))


In [ ]:
# Step 11: Save the Best Model
# joblib is standard for saving sklearn pipelines
# the saved file includes both the preprocessor AND the model in one object

joblib.dump(best_model, 'best_breast_cancer_model.pkl')

print("Saved: best_breast_cancer_model.pkl")
print(f"Model:    {best_model_name}")
print(f"Accuracy: {best_accuracy*100:.2f}%")
